In [16]:
# Model1: XGBoost model to predict Total Alkalinity (TA)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
import xgboost as xgb  # pip install xgboost


In [17]:
# Load engineered training features and join with TA target

features_path = "../New Datasets/Combined/combined_training_engineered.csv"
water_quality_path = "../Provided Datasets/water_quality_training_dataset.csv"

combined = pd.read_csv(features_path)
water_quality = pd.read_csv(water_quality_path)

# Standardize join keys to match `combined`
water_quality_std = water_quality.rename(
    columns={
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Sample Date": "sample_date",
    }
)

# Keep only join keys + TA target
ta_target = water_quality_std[["latitude", "longitude", "sample_date", "Total Alkalinity"]]

# Inner join to align features with TA labels
full = combined.merge(ta_target, on=["latitude", "longitude", "sample_date"], how="inner")

print("Features shape (combined):", combined.shape)
print("Water quality shape:", water_quality.shape)
print("Joined training shape:", full.shape)
full.head()


Features shape (combined): (9319, 82)
Water quality shape: (9319, 6)
Joined training shape: (9319, 83)


,latitude,longitude,sample_date,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,...,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious,Total Alkalinity
0,-34.405833,19.600556,01-10-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,4703.580299,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,54.181
1,-34.405833,19.600556,02-08-2011,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1984.043203,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,36.247
2,-34.405833,19.600556,02-12-2015,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,6827.433216,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,57.800
3,-34.405833,19.600556,03-07-2013,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1466.619596,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,29.719
4,-34.405833,19.600556,03-09-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,2919.631339,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,58.231


In [18]:
# Build feature matrix X and target y for TA, then reduce multicollinearity

# Columns to exclude from features
exclude_cols = {
    "Total Alkalinity",                 # target
    "Electrical Conductance",           # do not include
    "Dissolved Reactive Phosphorus",    # do not include
    "latitude", "longitude",           # do not include
    "sample_date",                      # string key, not a feature
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["Total Alkalinity"]

print("Initial number of features:", len(base_feature_cols))

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# Expose reduced set for feature selection
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])


Initial number of features: 79
Dropped 24 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 55
Example remaining feature columns: ['gaia_changed_ever_frac', 'gaia_recent_change_5y_frac', 'gsw_change', 'gsw_occurrence', 'gsw_transitions', 'nir', 'green', 'swir16', 'NDMI', 'MNDWI']


In [19]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 90% of total importance, but ensure at least 20 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])


Total features after multicollinearity reduction: 55
Selected features after importance-based selection: 26
Top selected features: ['esa_flooded_frac_1km', 'esa_sparse_veg_frac_1km', 'esa_forest_frac_1km', 'gsw_occurrence_mean_1km', 'water_perm', 'soil', 'esa_shrub_frac_1km', 'recurrence_ratio', 'gsw_recurrence_mean_1km', 'esa_urban_frac_1km', 'esa_cropland_frac_1km', 'esa_grass_frac_1km', 'esa_lccs_class', 'gaia_changed_ever_frac_1km', 'esa_water_frac_1km']


In [20]:
# Baseline XGBoost model (for quick R² and importances)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Baseline Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Baseline Test  R²: {r2_score(y_test, y_test_pred):.3f}")

importances_base = model.feature_importances_
fi_base = pd.DataFrame({"feature": feature_cols, "importance": importances_base})
fi_base.sort_values("importance", ascending=False).head(20)


Baseline Train R²: 0.932
Baseline Test  R²: 0.835


,feature,importance
0,esa_flooded_frac_1km,0.096131
12,esa_lccs_class,0.081654
2,esa_forest_frac_1km,0.078833
1,esa_sparse_veg_frac_1km,0.069322
8,gsw_recurrence_mean_1km,0.065272
3,gsw_occurrence_mean_1km,0.062747
4,water_perm,0.055260
6,esa_shrub_frac_1km,0.049985
7,recurrence_ratio,0.049860
11,esa_grass_frac_1km,0.044285


In [23]:
# Stratified K-Fold + Optuna hyperparameter tuning for TA model

n_bins = 10
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-02-24 17:33:27,527] A new study created in memory with name: no-name-fe4b7936-c046-422b-b62d-e9875f604e6f
  0%|          | 0/40 [00:00<?, ?it/s]

[W 2026-02-24 17:33:27,533] Trial 0 failed with parameters: {'n_estimators': 315, 'max_depth': 3, 'learning_rate': 0.013097934359765058, 'subsample': 0.7453805941637446, 'colsample_bytree': 0.7598120667291092, 'min_child_weight': 8.457886945229523, 'gamma': 2.281382771974827, 'reg_alpha': 0.058446792605418074, 'reg_lambda': 2.8967792911798154} because of the following error: TypeError("XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'").
Traceback (most recent call last):
  File "/Users/ethan/Library/Python/3.10/lib/python/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/mc/q08d_s811l54wr44n0f4yhnw0000gp/T/ipykernel_97631/90705670.py", line 30, in objective
    model.fit(
  File "/Users/ethan/Library/Python/3.10/lib/python/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'
[W 2026

TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:
# Train final TA model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final TA model Train R²: {r2_train:.3f}")
print(f"Final TA model Test  R²: {r2_test:.3f}")

importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting TA (tuned model):")
print(fi_tuned.head(20).to_string(index=False))

fi_tuned.head(20)


Final TA model Train R²: 0.963
Final TA model Test  R²: 0.849

Top 20 most important features for predicting TA (tuned model):
                   feature  importance
      esa_flooded_frac_1km    0.111916
   esa_sparse_veg_frac_1km    0.109672
       esa_forest_frac_1km    0.097501
   gsw_recurrence_mean_1km    0.065537
   gsw_occurrence_mean_1km    0.056605
                water_perm    0.042542
        esa_shrub_frac_1km    0.041419
            esa_lccs_class    0.040858
          recurrence_ratio    0.036546
                      soil    0.035879
        esa_grass_frac_1km    0.030982
      gsw_occ_x_impervious    0.030629
gaia_changed_ever_frac_1km    0.030563
     esa_cropland_frac_1km    0.029182
        esa_water_frac_1km    0.028071
        esa_other_frac_1km    0.027548
        esa_urban_frac_1km    0.021166
          esa_change_count    0.016498
                gsw_change    0.015208
    gaia_changed_ever_frac    0.014116


,feature,importance
0,esa_flooded_frac_1km,0.111916
1,esa_sparse_veg_frac_1km,0.109672
2,esa_forest_frac_1km,0.097501
8,gsw_recurrence_mean_1km,0.065537
3,gsw_occurrence_mean_1km,0.056605
4,water_perm,0.042542
6,esa_shrub_frac_1km,0.041419
12,esa_lccs_class,0.040858
7,recurrence_ratio,0.036546
5,soil,0.035879


In [ ]:
# Predict TA for validation rows and overwrite TA column in submission1.csv

# Load submission1.csv (with EC predictions) and validation engineered features
submission_path = "../submission1.csv"
submission = pd.read_csv(submission_path)
val_features = pd.read_csv("../New Datasets/Combined/combined_validation_engineered.csv")

# Standardize keys in submission to match engineered features
sub_std = submission.rename(columns={
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Sample Date": "sample_date",
})

# Join validation features with submission IDs
val_full = val_features.merge(
    sub_std[["latitude", "longitude", "sample_date"]],
    on=["latitude", "longitude", "sample_date"],
    how="inner",
)

print("Validation rows with matched features (TA):", val_full.shape[0])

# Build X_val using same TA feature set
X_val = val_full[feature_cols].copy()

ta_pred = final_model.predict(X_val)

# Map predictions back into submission
pred_df = val_full[["latitude", "longitude", "sample_date"]].copy()
pred_df["Total Alkalinity"] = ta_pred
pred_df = pred_df.rename(columns={
    "latitude": "Latitude",
    "longitude": "Longitude",
    "sample_date": "Sample Date",
})

submission = submission.drop(columns=["Total Alkalinity"]).merge(
    pred_df,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left",
)

submission.to_csv(submission_path, index=False)
print("Updated submission1.csv with Total Alkalinity predictions.")
submission.head()


Validation rows with matched features (TA): 200
Updated submission1.csv with Total Alkalinity predictions.


,Latitude,Longitude,Sample Date,Electrical Conductance,Dissolved Reactive Phosphorus,Total Alkalinity
0,-32.043333,27.822778,01-09-2014,503.45038,45.733494,138.076416
1,-33.329167,26.077500,16-09-2015,194.69760,15.421585,20.969130
2,-32.991639,27.640028,07-05-2015,233.47496,17.202000,39.641472
3,-34.096389,24.439167,07-02-2012,272.83110,17.007240,82.187828
4,-32.000556,28.581667,01-10-2014,570.31760,27.204762,117.721016


In [ ]:
df = pd.read_csv("../submission1.csv")

# Put Longitude first, then Latitude, then keep the rest in the same order
cols = df.columns.tolist()
fixed_first = ["Longitude", "Latitude", "Sample Date"]
rest = [c for c in cols if c not in fixed_first]
df = df[fixed_first + rest]

df.to_csv("../submission1.csv", index=False)